# HWD — Hierarchical Wavelet Diffusion
Imputasi data deret waktu multivariate dengan Wavelet Conditioning + Conditional Diffusion Model.

**Urutan eksekusi:** jalankan cell dari atas ke bawah secara berurutan. Jangan skip kecuali disebutkan opsional.

## Cell 1 — Install Library
Jalankan sekali. Restart runtime jika diminta.

In [ ]:
!pip install PyWavelets properscoring --quiet
print('✓ Install selesai')


✓ Install selesai


## Cell 2 — Setup: Mount Drive & Path
`DATASET` bisa diganti: `'kdd'` | `'guangzhou'` | `'physio'` | `'imk'`

In [ ]:
import os, sys, time, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output, display
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ── Path konfigurasi ──────────────────────────────────────────────────────────
PROJECT_DIR = '/content/drive/MyDrive/Skripsi/HWD'
DATA_DIR    = '/content/drive/MyDrive/Skripsi/HWD/Data'
CKPT_DIR    = '/content/drive/MyDrive/Skripsi/HWD/Checkpoints'
RESULT_DIR  = '/content/drive/MyDrive/Skripsi/HWD/Results'

for d in [CKPT_DIR, RESULT_DIR]: os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

DATASET      = 'kdd'    # ← ganti dataset di sini
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RATES        = [0.4]
SEED         = 3407

print(f'CWD    : {os.getcwd()}')
print(f'Device : {DEVICE}')
print(f'Dataset: {DATASET}')
print(f'Rates  : {RATES}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CWD    : /content/drive/.shortcut-targets-by-id/1zRoo-7RDvSObSFdvqYlY2mLWJnkFJw8u/HWD
Device : cuda
Dataset: kdd
Rates  : [0.4]


## Cell 3 — Preprocessing
Menggunakan fungsi dari `train.py`. Skip otomatis jika file `_norm.csv` sudah ada.

> **Wajib hapus `KDD_norm.csv` lama** jika ingin re-generate dengan sentinel+outlier treatment baru.

In [ ]:
import train

# Preprocessing — skip otomatis kalau file sudah ada
# Untuk re-generate: hapus dulu KDD_norm.csv dari Data/
if DATASET == 'kdd':
    train.preprocess_kdd(
        raw=f'{DATA_DIR}/KDD.csv',
        out=f'{DATA_DIR}/KDD_norm.csv'
    )
elif DATASET == 'imk':
    train.preprocess_imk(
        raw=f'{DATA_DIR}/IMK_raw.csv',
        out=f'{DATA_DIR}/IMK_norm.csv'
    )
else:
    norm_map = {'guangzhou': 'Guangzhou_norm.csv', 'physio': 'Physio_norm.csv'}
    raw_map  = {'guangzhou': 'guangzhou.csv',       'physio': 'physio.csv'}
    train.preprocess_generic(
        raw_csv = f'{DATA_DIR}/{raw_map[DATASET]}',
        out_csv = f'{DATA_DIR}/{norm_map[DATASET]}',
        prefix  = DATASET.capitalize()
    )


Skip — /content/drive/MyDrive/Skripsi/HWD/Data/KDD_norm.csv sudah ada


## Cell 4 — Generate Missing Masks
Buat mask MCAR/MAR/MNAR/block untuk semua rate. Skip otomatis jika mask sudah ada.

In [ ]:
import generate_missing as gm

if DATASET in ('kdd', 'guangzhou', 'physio'):
    gm.generate_all_masks(DATASET, ['mcar','mar','mnar'], RATES, [SEED])
else:
    gm.generate_all_masks('imk', ['mcar','block'], RATES, [SEED],
                          data_path='Data/IMK_norm.csv', block_size=2)

Loading Data/KDD_norm.csv ...
  Shape: (8035, 99)
  Skip (exists): kdd_0.4_3407.csv
  Skip (exists): kddmar_0.4_3407.csv
  Saved: kddmnar_0.4_3407.csv  (actual missing: 0.400)

Done. 1 file mask baru dibuat di Data/mask/kdd/


## Cell 5 — Konfigurasi Model
Hyperparameter HWD. Untuk eksperimen: hanya ubah `RATES` di Cell 2.

In [ ]:
PARAMS = {
    'kdd'      : {'seq_len': 48, 'enc_in': 99,  'c_out': 99},
    'guangzhou': {'seq_len': 48, 'enc_in': 214, 'c_out': 214},
    'physio'   : {'seq_len': 48, 'enc_in': 37,  'c_out': 37},
    'imk'      : {'seq_len': 12, 'enc_in': 0,   'c_out': 0},   # enc_in auto dari data
}
p = PARAMS[DATASET]

BASE_CFG = {
    'dataset'           : DATASET,
    'seed'              : SEED,
    'seq_len'           : p['seq_len'],
    'enc_in'            : p['enc_in'],
    'c_out'             : p['c_out'],
    'd_model'           : 128,
    'e_layers'          : 4,
    'nheads'            : 8,
    'channel'           : 128,
    'proj_t'            : 128,
    'residual_layers'   : 4,
    'timeemb'           : 128,
    'featureemb'        : 16,
    'diffusion_step_num': 50,
    'schedule'          : 'quad',
    'beta_start'        : 0.0001,
    'beta_end'          : 0.2,
    'epoch_diff'        : 400,
    'learning_rate_diff': 1e-3,
    'mask_ratio_ssl'    : 0.2,
    'avg_mask_len_ssl'  : 3,
    'wavelet'           : 'db4',
    'levels'            : 3,
    'batch'             : 16,
    'device'            : DEVICE,
    'n_samples'         : 100,
    'save_dir'          : CKPT_DIR,
    'data_path'         : f'{DATA_DIR}/IMK_norm.csv',
    'col_names_path'    : '',
    'exp_mask_path'     : '',
}

configs = train.get_config(BASE_CFG)
print(f'Dataset : {configs.dataset}  |  Device : {configs.device}')
print(f'Epochs  : {configs.epoch_diff}  |  Wavelet : {configs.wavelet} L={configs.levels}')
print(f'Model   : d_model={configs.d_model}, e_layers={configs.e_layers}, residual={configs.residual_layers}')
print(f'Diffusion: T={configs.diffusion_step_num}, beta=[{configs.beta_start},{configs.beta_end}], sched={configs.schedule}')


Dataset : kdd  |  Device : cuda
Epochs  : 400  |  Wavelet : db4 L=3
Model   : d_model=128, e_layers=4, residual=4
Diffusion: T=50, beta=[0.0001,0.2], sched=quad


## Cell 6 — Training + Evaluasi Semua Missing Rate
- Skip training jika checkpoint sudah ada
- Loss curve disimpan per rate
- Evaluasi MAE / RMSE / CRPS otomatis setelah training

> **Estimasi waktu:** ~30 menit per rate di T4 GPU

In [ ]:
import dataset as ds_module
from models import main_model

np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed(SEED)

all_results   = {}
all_losses    = {}
trained_models = {}  # simpan model di memory untuk testing & full inference

# ── Helper: training dengan loss tracking + resume ─────────────────────────
def train_with_loss_tracking(configs_r):
    import time
    from torch import optim

    train_loader, _ = ds_module.get_dataset(configs_r)
    model_r = main_model.HWD(configs_r).to(configs_r.device)
    optimizer = optim.Adam(model_r.parameters(),
                           lr=configs_r.learning_rate_diff, weight_decay=1e-6)
    p1 = int(0.75 * configs_r.epoch_diff)
    p2 = int(0.90 * configs_r.epoch_diff)
    sched = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[p1, p2], gamma=0.1)

    ckpt       = f"{configs_r.save_dir}/hwd_{configs_r.dataset}_mr{configs_r.missing_rate}.pt"
    meta_path  = ckpt.replace('.pt', '_meta.txt')
    loss_path  = ckpt.replace('.pt', '_losses.npy')
    start_epoch = 0

    if os.path.exists(ckpt):
        model_r.load_state_dict(torch.load(ckpt, map_location=configs_r.device))
        if os.path.exists(meta_path):
            with open(meta_path) as mf:
                start_epoch = int(mf.read().strip())
        if start_epoch >= configs_r.epoch_diff:
            print(f'  Sudah epoch {start_epoch}, skip training.')
            existing = np.load(loss_path).tolist() if os.path.exists(loss_path) else []
            return model_r, existing
        print(f'  Resume dari epoch {start_epoch} → {configs_r.epoch_diff}')

    losses = np.load(loss_path).tolist() if os.path.exists(loss_path) else []

    for epoch in range(start_epoch, configs_r.epoch_diff):
        model_r.train()
        epoch_loss = []
        t0 = time.time()
        for obs_d, obs_m, obs_tp, gt_m in train_loader:
            optimizer.zero_grad()
            loss = model_r(obs_d, obs_m, obs_tp, gt_m)
            loss.backward()
            optimizer.step()
            epoch_loss.append(loss.item())
        sched.step()
        avg = np.mean(epoch_loss)
        losses.append(avg)
        if epoch % 50 == 0 or epoch == configs_r.epoch_diff - 1:
            print(f'  Epoch {epoch+1:>4} | time {time.time()-t0:.1f}s | loss {avg:.6f}')

    os.makedirs(configs_r.save_dir, exist_ok=True)
    torch.save(model_r.state_dict(), ckpt)
    with open(meta_path, 'w') as mf:
        mf.write(str(configs_r.epoch_diff))
    np.save(loss_path, np.array(losses))
    print(f'  Checkpoint: {ckpt} (epoch {configs_r.epoch_diff})')
    return model_r, losses


# ═══════════════════════════════════════════════════════
# FASE 1: TRAINING semua rate
# ═══════════════════════════════════════════════════════
print('=' * 50)
print('FASE 1: Training')
print('=' * 50)

for rate in RATES:
    print(f"\n{'='*45}\n  Missing rate : {rate}\n{'='*45}")
    cfg_r = train.get_config({**vars(configs), 'missing_rate': rate})
    model_r, losses = train_with_loss_tracking(cfg_r)
    trained_models[rate] = model_r
    all_losses[rate]     = losses

print('\n✓ Semua training selesai.')


# ═══════════════════════════════════════════════════════
# FASE 2: TESTING semua rate (evaluasi metrik)
# ═══════════════════════════════════════════════════════
print('\n' + '=' * 50)
print('FASE 2: Testing (Evaluasi Metrik)')
print('=' * 50)

for rate in RATES:
    print(f"\n{'='*45}\n  Missing rate : {rate}\n{'='*45}")
    cfg_r   = train.get_config({**vars(configs), 'missing_rate': rate})
    model_r = trained_models[rate]

    hwd_r = train.diffusion_test(cfg_r, model_r)

    # Mean imputation baseline
    train_loader_r, test_loader_r = ds_module.get_dataset(cfg_r)
    sum_v, sum_c = None, None
    for d_, m_, _, _ in train_loader_r:
        v = (d_ * m_).sum(dim=(0,1))
        c = m_.sum(dim=(0,1))
        sum_v = v if sum_v is None else sum_v + v
        sum_c = c if sum_c is None else sum_c + c
    col_means = sum_v / (sum_c + 1e-8)

    mae_t = rmse_t = pts = 0
    for d_, m_, _, gt_ in test_loader_r:
        B, L, K = d_.shape
        ev   = (gt_ - m_).clamp(0, 1)
        imp  = d_ * m_ + col_means.unsqueeze(0).unsqueeze(0).expand(B,L,K) * (1-m_)
        idx  = torch.where(ev == 1)
        diff = (imp[idx] - d_[idx]).abs()
        mae_t  += diff.sum().item()
        rmse_t += (diff**2).sum().item()
        pts    += len(idx[0])

    all_results[rate] = {
        'hwd' : hwd_r,
        'mean': {'MAE': mae_t/(pts+1e-8), 'RMSE': (rmse_t/(pts+1e-8))**0.5, 'CRPS': float('nan')}
    }

# Ringkasan metrik
print('\n' + '=' * 50)
print('RINGKASAN METRIK')
print('=' * 50)
print(f"  {'Rate':<8} {'MAE HWD':<12} {'RMSE HWD':<12} {'CRPS HWD':<12}")
print(f"  {'-'*44}")
for rate in RATES:
    r = all_results[rate]['hwd']
    print(f"  {rate:<8} {r['MAE']:<12.4f} {r['RMSE']:<12.4f} {r['CRPS']:<12.4f}")
print('\n✓ Semua testing selesai.')


# ═══════════════════════════════════════════════════════
# FASE 3: FULL INFERENCE semua rate (output operasional)
# Dijalankan setelah semua training & testing selesai
# ═══════════════════════════════════════════════════════
print('\n' + '=' * 50)
print('FASE 3: Full Inference (Output Operasional)')
print('=' * 50)

for rate in RATES:
    cfg_r   = train.get_config({**vars(configs), 'missing_rate': rate})
    model_r = trained_models[rate]
    print(f'\n  Full inference mr={rate}...')
    train.full_inference_and_save(cfg_r, model_r)

print('\n✓ Semua fase selesai.')



  Missing rate : 0.4
  Epoch    1 | time 6.4s | loss 1.007787
  Epoch   51 | time 4.8s | loss 0.376265
  Epoch  101 | time 4.7s | loss 0.307896
  Epoch  151 | time 4.7s | loss 0.282326
  Epoch  201 | time 4.7s | loss 0.259248
  Epoch  251 | time 4.7s | loss 0.185497
  Epoch  301 | time 4.8s | loss 0.217850
  Epoch  351 | time 4.8s | loss 0.218334
  Epoch  400 | time 4.8s | loss 0.188238
  Checkpoint: /content/drive/MyDrive/Skripsi/HWD/Checkpoints/hwd_kdd_mr0.4.pt
Test batches: 4
  batch 1 | elapsed 1265.9s
  batch 2 | elapsed 1265.1s
  batch 3 | elapsed 1265.0s
  batch 4 | elapsed 258.1s
Imputation saved: HWD_Imputation_kdd_mr0.4_original.csv  min=-18618.207  max=999017.130
  HWD  MAE=0.1740  RMSE=0.4653  CRPS=0.1962
[full_inference] Running on all 167 windows (6 batches)...


KeyboardInterrupt: 

## Cell 7 — Tabel Hasil Evaluasi

In [ ]:
# Baseline FGTI dari paper (KDD MCAR)
FGTI_REF = {0.1: {'MAE':0.149,'RMSE':0.406,'CRPS':0.158},
             0.2: {'MAE':0.163,'RMSE':0.432,'CRPS':0.174},
             0.3: {'MAE':0.183,'RMSE':0.456,'CRPS':0.198},
             0.4: {'MAE':0.210,'RMSE':0.484,'CRPS':0.228}}
CSDI_REF  = {0.1: {'MAE':0.177,'RMSE':0.459,'CRPS':0.224},
             0.2: {'MAE':0.191,'RMSE':0.471,'CRPS':0.237}}

W = 65
print(f'\n{"═"*W}')
print(f'  Dataset : {DATASET.upper()} | Mechanism: MCAR')
print(f'{"─"*W}')
print(f'  {"Rate":<6} {"Model":<22} {"MAE":>8} {"RMSE":>8} {"CRPS":>8}')
print(f'{"─"*W}')

for rate in RATES:
    if rate not in all_results: continue
    res = all_results[rate]

    def row(name, d):
        crps = f"{d.get('CRPS',float('nan')):>8.4f}" if not (d.get('CRPS') != d.get('CRPS')) else f"{'N/A':>8}"
        print(f'  {str(rate):<6} {name:<22} {d["MAE"]:>8.4f} {d["RMSE"]:>8.4f} {crps}')

    row('Mean Imputation',  res['mean'])
    if rate in CSDI_REF:  row('CSDI (paper)',    CSDI_REF[rate])
    if rate in FGTI_REF:  row('FGTI (paper)',    FGTI_REF[rate])
    row('HWD (ours)',       res['hwd'])
    print(f'  {"─"*63}')

print(f'{"═"*W}')


## Cell 8 — Loss Curve per Epoch
Satu subplot per missing rate. Menampilkan tren konvergensi training.

In [ ]:
rates_with_loss = [r for r in RATES if r in all_losses and len(all_losses[r]) > 0]
if not rates_with_loss:
    print("Belum ada loss data — jalankan Cell 6 dulu.")
else:
    n = len(rates_with_loss)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=False)
    if n == 1: axes = [axes]

    for ax, rate in zip(axes, rates_with_loss):
        losses = all_losses[rate]
        epochs = range(1, len(losses)+1)
        ax.plot(epochs, losses, color='#2d6ebb', lw=1.5)

        # Moving average
        window = max(1, len(losses)//20)
        if len(losses) >= window:
            ma = np.convolve(losses, np.ones(window)/window, mode='valid')
            ax.plot(range(window, len(losses)+1), ma,
                    color='#e05c2d', lw=2.5, label=f'MA-{window}')

        ax.set_title(f'mr = {rate}', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss' if ax == axes[0] else '')
        ax.set_xlim(1, len(losses))
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        # Annotasi loss akhir
        ax.annotate(f'final: {losses[-1]:.4f}',
                    xy=(len(losses), losses[-1]),
                    xytext=(-40, 10), textcoords='offset points',
                    fontsize=8, color='#e05c2d',
                    arrowprops=dict(arrowstyle='->', color='#e05c2d', lw=1))

    fig.suptitle(f'Training Loss — {DATASET.upper()}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_loss = f'{RESULT_DIR}/loss_curve_{DATASET}.png'
    plt.savefig(out_loss, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved: {out_loss}')


## Cell 10 — Visualisasi Imputasi
- **Plot A:** Time-series imputasi vs ground truth + confidence interval untuk tiap rate
- **Plot B:** Before vs After: distribusi beberapa variabel sebelum dan sesudah imputasi
- **Plot C:** Summary statistik missing → imputed per variabel

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Pilih feature yang akan divisualisasi ────────────────────────────────────
FEAT_IDX = 0   # ← ganti untuk variabel yang berbeda

# ════════════════════════════════════════════════════════
# PLOT A — Time-series imputasi per missing rate
# ════════════════════════════════════════════════════════
rates_done = [r for r in RATES if r in all_results]
n_rates    = len(rates_done)

fig, axes = plt.subplots(n_rates, 1, figsize=(14, 4 * n_rates), sharex=False)
if n_rates == 1: axes = [axes]

for ax, rate in zip(axes, rates_done):
    cfg_r     = train.get_config({**vars(configs), 'missing_rate': rate})
    _, test_l = ds_module.get_dataset(cfg_r)
    ckpt_v    = f"{CKPT_DIR}/hwd_{DATASET}_mr{rate}.pt"

    model_v = main_model.HWD(cfg_r).to(cfg_r.device)
    model_v.load_state_dict(torch.load(ckpt_v, map_location=cfg_r.device))
    model_v.eval()

    # Ambil batch pertama
    obs_d, obs_m, obs_tp, gt_m = next(iter(test_l))
    with torch.no_grad():
        out = model_v.evaluate(obs_d, obs_m, obs_tp, gt_m, n_samples=50)
    imp_samps, c_target, eval_pts, _, _ = out

    # [B, n_samples, K, L] → permute → [B, L, K]
    imp_samps = imp_samps.permute(0, 1, 3, 2)
    c_target  = c_target.permute(0, 2, 1)
    eval_pts  = eval_pts.permute(0, 2, 1)

    # Pilih sample window terbaik (yang punya paling banyak missing)
    n_miss_per_window = (gt_m - obs_m).clamp(0,1).sum(dim=(1,2))
    b_idx = n_miss_per_window.argmax().item()

    truth    = c_target[b_idx, :, FEAT_IDX].cpu().numpy()
    ev_mask  = eval_pts[b_idx, :, FEAT_IDX].cpu().numpy()
    samps    = imp_samps[b_idx, :, :, FEAT_IDX].cpu().numpy()  # [n_samples, L]
    imp_med  = np.median(samps, axis=0)

    T    = len(truth)
    lo5  = np.percentile(samps, 5,  axis=0)
    hi95 = np.percentile(samps, 95, axis=0)
    lo25 = np.percentile(samps, 25, axis=0)
    hi75 = np.percentile(samps, 75, axis=0)

    # Plot
    ax.plot(range(T), truth,   'k-',  lw=2,   zorder=4, label='Ground truth')
    ax.plot(range(T), imp_med, '--',  lw=1.8, color='#3a86ff', zorder=3, label='HWD median')
    ax.fill_between(range(T), lo5,  hi95, alpha=0.15, color='#3a86ff', label='5–95%')
    ax.fill_between(range(T), lo25, hi75, alpha=0.3,  color='#3a86ff', label='25–75%')

    miss_idx = np.where(ev_mask == 1)[0]
    if len(miss_idx):
        ax.scatter(miss_idx, truth[miss_idx],
                   c='#e05c2d', s=40, zorder=5, label=f'Missing ({len(miss_idx)} pts)')

    ax.set_title(f'Missing Rate = {rate} | Feature idx {FEAT_IDX}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Z-score')
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.25)

fig.suptitle(f'HWD Imputasi — {DATASET.upper()}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
out_a = f'{RESULT_DIR}/viz_timeseries_{DATASET}.png'
plt.savefig(out_a, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {out_a}')


## Cell 11 — Before vs After Imputasi
Perbandingan distribusi data sebelum imputasi (dengan NaN) vs sesudah imputasi, untuk 4 variabel utama.

In [ ]:
# Ambil data asli dan imputasi rate pertama untuk perbandingan
rate_ba  = rates_done[0]
denorm_path = f'{RESULT_DIR}/HWD_Imputation_{DATASET}_mr{rate_ba}_denorm.csv'
raw_path    = f'{DATA_DIR}/KDD.csv'   # ganti untuk dataset lain

if not os.path.exists(denorm_path):
    print(f'File denorm belum ada: {denorm_path}\nJalankan Cell 9 dulu.')
elif not os.path.exists(raw_path):
    print(f'File raw tidak ditemukan: {raw_path}')
else:
    df_raw = pd.read_csv(raw_path)
    df_imp = pd.read_csv(denorm_path)

    num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()

    # Pilih 4 variabel dengan % missing tertinggi
    miss_pct = df_raw[num_cols].isna().mean().sort_values(ascending=False)
    top4_cols = miss_pct.head(4).index.tolist()

    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    colors = {'raw': '#999999', 'imp': '#3a86ff'}

    for col_i, col in enumerate(top4_cols):
        if col not in df_imp.columns:
            continue
        raw_vals = df_raw[col].dropna().values
        imp_vals = df_imp[col].dropna().values

        # Baris 0: histogram overlay
        ax0 = axes[0, col_i]
        ax0.hist(raw_vals, bins=40, alpha=0.55, color=colors['raw'],
                 density=True, label=f'Before\n(n={len(raw_vals):,})')
        ax0.hist(imp_vals, bins=40, alpha=0.55, color=colors['imp'],
                 density=True, label=f'After\n(n={len(imp_vals):,})')
        ax0.set_title(col.replace('_', '\n'), fontsize=8, fontweight='bold')
        ax0.set_ylabel('Density' if col_i == 0 else '')
        ax0.legend(fontsize=7)
        ax0.grid(True, alpha=0.2)

        # Baris 1: boxplot side by side
        ax1 = axes[1, col_i]
        bp = ax1.boxplot([raw_vals, imp_vals],
                          labels=['Before', 'After'],
                          patch_artist=True, widths=0.5,
                          medianprops=dict(color='black', lw=2))
        bp['boxes'][0].set_facecolor(colors['raw'])
        bp['boxes'][1].set_facecolor(colors['imp'])
        for patch in bp['boxes']:
            patch.set_alpha(0.6)
        ax1.set_title(f'IQR comparison', fontsize=8)
        ax1.grid(True, alpha=0.2, axis='y')

        # Statistik ringkas
        n_imp = df_raw[col].isna().sum()
        ax1.text(0.5, 0.97,
                 f'Imputed: {n_imp:,} cells ({n_imp/len(df_raw)*100:.1f}%)',
                 transform=ax1.transAxes, ha='center', va='top',
                 fontsize=7, color='#e05c2d')

    fig.suptitle(
        f'Before vs After Imputasi — {DATASET.upper()} | rate={rate_ba} | Top-4 variabel missing',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    out_b = f'{RESULT_DIR}/viz_before_after_{DATASET}_mr{rate_ba}.png'
    plt.savefig(out_b, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved: {out_b}')


## Cell 12 — Summary Statistik Imputasi
Tabel ringkas: jumlah sel yang diimputasi dan perubahan statistik deskriptif per missing rate.

In [ ]:
rows = []
for rate in rates_done:
    path = f'{RESULT_DIR}/HWD_Imputation_{DATASET}_mr{rate}_denorm.csv'
    if not os.path.exists(path): continue
    df_imp = pd.read_csv(path)
    raw_path_chk = f'{DATA_DIR}/KDD.csv'
    if not os.path.exists(raw_path_chk): continue
    df_raw = pd.read_csv(raw_path_chk)
    num_c  = df_raw.select_dtypes(include=[np.number]).columns
    df_raw_n = df_raw[num_c]

    n_natural = df_raw_n.isna().sum().sum()
    n_total   = df_raw_n.size
    pct_nat   = n_natural / n_total * 100
    n_imp_cells = (df_raw_n.isna() & df_imp.notna()).sum().sum()                   if set(num_c).issubset(df_imp.columns) else 0

    rows.append({
        'Rate'         : rate,
        'Natural NaN'  : f'{n_natural:,} ({pct_nat:.1f}%)',
        'Cells Imputed': f'{n_imp_cells:,}',
        'Mean Before'  : f'{df_raw_n.stack().mean():.4f}',
        'Mean After'   : f'{df_imp[num_c].stack().mean():.4f}' if set(num_c).issubset(df_imp.columns) else 'N/A',
        'Std Before'   : f'{df_raw_n.stack().std():.4f}',
        'Std After'    : f'{df_imp[num_c].stack().std():.4f}' if set(num_c).issubset(df_imp.columns) else 'N/A',
        'MAE'          : f'{all_results[rate]["hwd"]["MAE"]:.4f}',
        'RMSE'         : f'{all_results[rate]["hwd"]["RMSE"]:.4f}',
        'CRPS'         : f'{all_results[rate]["hwd"]["CRPS"]:.4f}',
    })

if rows:
    df_sum = pd.DataFrame(rows).set_index('Rate')
    print('\nSummary Statistik Imputasi:')
    display(df_sum)
    df_sum.to_csv(f'{RESULT_DIR}/summary_{DATASET}.csv')
    print(f'✓ Saved: {RESULT_DIR}/summary_{DATASET}.csv')
else:
    print('Belum ada data — jalankan Cell 6, 9 dulu.')


## Cell 13 — Incremental Learning *(Opsional)*
Fine-tune model pada data late response baru tanpa training ulang dari awal.

In [ ]:
import incremental

BASE_CKPT    = f'{CKPT_DIR}/hwd_{DATASET}_mr{RATES[0]}.pt'
NEW_CSV_PATH = ''   # ← isi path CSV data baru (late response)

if not NEW_CSV_PATH:
    print('NEW_CSV_PATH belum diisi — skip incremental learning.')
else:
    configs_new   = train.get_config({**vars(configs), 'data_path': NEW_CSV_PATH})
    new_train_l, new_test_l = ds_module.get_dataset(configs_new)

    model_base = main_model.HWD(configs).to(configs.device)
    model_base.load_state_dict(torch.load(BASE_CKPT, map_location=configs.device))

    model_incr = incremental.incremental_finetune(
        model_base, new_train_l,
        freeze_ratio=0.8, lr=1e-5, epochs=50
    )
    incr_ckpt = BASE_CKPT.replace('.pt', '_incr.pt')
    torch.save(model_incr.state_dict(), incr_ckpt)

    _, old_test_l = ds_module.get_dataset(configs)
    result = incremental.run_forgetting_check(
        BASE_CKPT, incr_ckpt, configs, old_test_l, new_test_l
    )
    print(f'Forgetting  : {result["forgetting_pct"]:+.2f}%  (target < 5%)')
    print(f'Improvement : {result["improvement_pct"]:+.2f}%  (target > 0%)')
